# Banyan City — free Wan T2V rendering on Kaggle
Renders a node's `shots.md` prompts into per-beat clips with **open Apache-2.0
[Wan 2.1 T2V 1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers) weights** on Kaggle's
free GPU quota (30 h/week) — the tree's permanent $0 rendering floor, reproducible by any citizen
(**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator = **GPU
T4 x2**, Internet = ON, phone-verified account. **Run All** (no kernel restart needed — Kaggle's own torch is left untouched). ~20-45 min per 5s clip. Since loop cycle 007 an episode is 15-25 SHOTS (one per beat, 3-6s each,
camera on the referent — see SCRIPT-SPEC.md), so a full episode is ~8-15 GPU-hours: about one
episode per week inside the free 30 h quota. The notebook skips clips that already exist, so a
preempted session resumes by re-running. Output: `/kaggle/working/clips.zip` → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`.

Provenance: every clip gets a `meta.yaml` (§7.2). Output is 480×832 (9:16) 5s — the current best a
free T4 does; the season's canon quality bar is decided by the founder on material (R4/D8).

**Free-tier notes (learned on the first real run, 2026-07-25):** Kaggle's free instance is
RAM-poor (~13 GB) and VRAM-ok (T4 = 15 GB), so the 1.3B model is kept **resident on the GPU** —
`enable_model_cpu_offload()` streams weights through system RAM and kills the kernel. Torch is NEVER reinstalled (a fresh torchvision against the running torch
raises `INTERNAL ASSERT FAILED` in `Dtype.cpp`); diffusers goes in with `--no-deps` at >= 0.33,
which is the first version containing `WanPipeline`.
If the kernel still dies, drop `STEPS` to 30 and re-run: finished clips are skipped.

On a single free **T4 (14.56 GiB usable)** the model does not fit resident — Wan's UMT5-XXL text
encoder is the large part, not the 1.3B transformer — so the loader measures VRAM/RAM and picks
resident → model-offload → sequential-offload accordingly. Sequential offload is the slow floor:
expect the long end of the 20-45 min per clip. A shot that OOMs anyway retries once at 61 frames /
416x720 instead of losing the queue.


In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
NODE   = "001"        # any node id with a shots.md
BEATS  = [1]          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
STEPS  = 25            # 30 = faster/rougher, 50 = slower/cleaner
REPO_URL = "https://github.com/olegmlkvorg/banyan-city.git"


In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
# Do NOT reinstall torch. Kaggle ships a working torch/torchvision pair; the
# first real run (2026-07-25) tried to pin its own and hit INTERNAL ASSERT
# FAILED in Dtype.cpp — a fresh torchvision against the already-imported
# torch. Install diffusers with --no-deps so pip cannot pull a second torch in
# behind it. WanPipeline needs diffusers >= 0.33 (0.32 lacks it entirely —
# that was the failure after the pin).
%pip -q install --no-deps "diffusers==0.33.1"
%pip -q install ftfy imageio imageio-ffmpeg pyyaml psutil

# A kernel that already imported an older diffusers keeps it in memory no
# matter what pip writes to disk (this bit the founder on 2026-07-25: the
# session still held 0.32.2). Compare disk vs memory and say so plainly.
import sys
from importlib.metadata import version
on_disk = version("diffusers")
in_mem = getattr(sys.modules.get("diffusers"), "__version__", None)
print(f"diffusers on disk: {on_disk}" + (f" | already imported in this kernel: {in_mem}" if in_mem else ""))
if in_mem and in_mem != on_disk:
    print("\n*** STOP: this kernel is holding an older diffusers.\n"
          "    Run > Restart & clear cell outputs, then Run All again.\n"
          "    (Nothing is lost — finished clips are skipped on re-run.) ***\n")


# Kaggle's BATCH image ships a newer transformers than its interactive one, and
# `transformers.utils.FLAX_WEIGHTS_NAME` is gone from it. diffusers 0.33 imports
# that name at module load, so `from diffusers import WanPipeline` died with
# "cannot import name 'FLAX_WEIGHTS_NAME'" before touching the GPU (first batch
# push, 2026-07-25). The names are plain filename constants and nothing on the
# Wan path reads a flax/tf checkpoint, so define what is missing rather than
# repinning transformers — a repin drags tokenizers and risks the torch pair.
import transformers.utils as _tu
for _name, _val in (("FLAX_WEIGHTS_NAME", "flax_model.msgpack"),
                    ("TF2_WEIGHTS_NAME", "tf_model.h5"),
                    ("TF_WEIGHTS_NAME", "model.ckpt")):
    if not hasattr(_tu, _name):
        setattr(_tu, _name, _val)
        print(f"shimmed transformers.utils.{_name} (removed upstream)")

import pathlib
import subprocess
import sys

if not pathlib.Path("banyan-city").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
sys.path.insert(0, "banyan-city/pipeline")
from generate_shots import parse_shots
import yaml

node_dirs = [d for d in pathlib.Path(f"banyan-city/genomes/{GENOME}/nodes").iterdir() if d.is_dir()]
node_dir = next((d for d in sorted(node_dirs) if d.name.startswith(NODE)), None)
assert node_dir, f"no node dir starting with {NODE!r} — check NODE above"
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo:
    print(f"  {s['num']:02d} {s['slug']}")


In [ ]:
# ---- model: Wan 2.1 T2V 1.3B (Apache-2.0) ----------------------------------
# Two failures on 2026-07-25 taught the shape of this cell:
#   - enable_model_cpu_offload() on a RAM-poor box killed the kernel;
#   - .to("cuda") on a single T4 (14.56 GiB usable) hit CUDA OOM, because Wan
#     pairs a 1.3B video transformer with a UMT5-XXL text encoder — the text
#     encoder, not the transformer, is what does not fit.
# So: measure the machine, then pick the cheapest strategy that fits it.
import gc

import psutil
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator = GPU (needs phone verification)"
# Fail here, not 8 GiB of weights later. A batch session that lands on a Tesla
# P100 (sm_60) loads the whole pipeline and then dies inside an embedding lookup
# with "no kernel image is available for execution on the device", because the
# preinstalled torch ships no Pascal kernels (2026-07-25). Check the arch the
# device reports against the arch list torch was actually built for.
_cap = torch.cuda.get_device_capability(0)
_have = torch.cuda.get_arch_list()
if f"sm_{_cap[0]}{_cap[1]}" not in _have:
    raise SystemExit(
        f"{torch.cuda.get_device_name(0)} is sm_{_cap[0]}{_cap[1]}, and this torch "
        f"was built for {_have}. Set Accelerator to GPU T4 x2 (sm_75) — or push "
        f"with machine_shape=nvidiaTeslaT4x2 in kernel-metadata.json.")
import diffusers
assert tuple(int(x) for x in diffusers.__version__.split(".")[:2]) >= (0, 33), (
    f"diffusers is {diffusers.__version__} — WanPipeline needs >= 0.33. "
    "Run > Restart & clear cell outputs, then Run All again.")
from diffusers import WanPipeline
from diffusers.utils import export_to_video

vram = torch.cuda.get_device_properties(0).total_memory / 2**30
ram = psutil.virtual_memory().available / 2**30
print(f"{torch.cuda.get_device_name(0)}: {vram:.1f} GiB VRAM | {ram:.1f} GiB RAM available")

pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,      # memory-map shards; never hold two copies
)
# AutoencoderKLWan has no slicing/tiling API (unlike the SD/SDXL VAEs) — ask
# before calling, so a VAE without them is not a crash (2026-07-25).
for opt in ("enable_slicing", "enable_tiling"):
    fn = getattr(pipe.vae, opt, None)
    if callable(fn):
        fn()
        print(f"vae: {opt}()")

# Wan pairs a 1.3B video transformer with a UMT5-XXL text encoder, and the
# TEXT ENCODER is what does not fit: ~11 GiB of fp16 weights against a T4's
# 14.6 GiB. model_cpu_offload moves modules in one at a time and still died
# with CUBLAS_STATUS_ALLOC_FAILED on the first generation (2026-07-25) —
# cuBLAS could not get a workspace handle in what was left.
#
# So the prompts are encoded ONCE, up front, and the text encoder is then
# thrown off the GPU for good. What remains resident is the transformer plus
# the VAE (~3 GiB), which fits a T4 with room to spare — and being resident is
# also the fast configuration. Sequential offload would fit too, but at
# 30-60 min a clip it cannot finish an episode inside a session.
def encode_all(shots, neg):
    """{prompt: (embeds, neg_embeds)} computed with the text encoder on GPU."""
    pipe.text_encoder.to("cuda")
    cache = {}
    with torch.no_grad():
        for s in shots:
            if s["prompt"] in cache:
                continue
            pe, ne = pipe.encode_prompt(prompt=s["prompt"], negative_prompt=neg,
                                        do_classifier_free_guidance=True,
                                        device="cuda")
            cache[s["prompt"]] = (pe, ne)
    pipe.text_encoder.to("cpu")
    gc.collect(); torch.cuda.empty_cache()
    return cache

NEG = "photorealistic, 3d render, text, watermark, low quality, blurry"
strategy = None
try:
    EMBEDS = encode_all(todo, NEG)
    pipe.transformer.to("cuda"); pipe.vae.to("cuda")
    strategy = f"text encoder freed after encoding {len(EMBEDS)} prompt(s); transformer+vae resident"
except Exception as e:
    # any API drift in encode_prompt, or a machine too small — say so and fall
    # back rather than failing the whole session
    print(f"pre-encode unavailable ({type(e).__name__}: {e}); falling back to offload")
    EMBEDS = None
    gc.collect(); torch.cuda.empty_cache()
    if ram >= 18:
        pipe.enable_model_cpu_offload(); strategy = "model cpu offload (per-module)"
    else:
        pipe.enable_sequential_cpu_offload(); strategy = "sequential cpu offload"
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"pipeline ready — {strategy}; {free/2**30:.1f} of {total/2**30:.1f} GiB VRAM free")


In [ ]:
# ---- generate: 480x832 (9:16), 81 frames @ 16fps = ~5s per beat ------------
# A batch session is capped at ~12 h and a 20-shot episode can reach it. Two
# habits so a timeout costs one clip and not the whole run: re-zip after every
# clip (a killed kernel never reaches the packing cell), and print the elapsed
# time per shot so the remote log says whether the queue can finish.
import shutil
import time
from datetime import date
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)
for s in todo:
    dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
    if dest.exists():
        print(f"skip {dest.name} (exists)"); continue
    t0 = time.time()
    print(f"beat {s['num']:02d} ({s['slug']}) …", flush=True)
    g = torch.Generator(device="cpu").manual_seed(SEED + s["num"])

    def generate(h, w, n):
        kw = dict(height=h, width=w, num_frames=n,
                  num_inference_steps=STEPS, generator=g)
        if EMBEDS is not None:
            pe, ne = EMBEDS[s["prompt"]]
            return pipe(prompt_embeds=pe, negative_prompt_embeds=ne, **kw).frames[0]
        return pipe(prompt=s["prompt"], negative_prompt=NEG, **kw).frames[0]

    try:
        frames = generate(832, 480, 81)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        # CUBLAS_STATUS_ALLOC_FAILED is an out-of-memory wearing a different
        # hat: cuBLAS asking for a workspace in VRAM that is already gone.
        if not isinstance(e, torch.cuda.OutOfMemoryError) and \
           not any(k in str(e) for k in ("CUBLAS", "out of memory", "CUDA error")):
            raise
        # one retry at 3s / lower res rather than losing the whole queue
        print(f"   {type(e).__name__} — retrying this shot at 61 frames, 416x720",
              flush=True)
        gc.collect(); torch.cuda.empty_cache()
        frames = generate(720, 416, 61)
    export_to_video(frames, str(dest), fps=16)
    shutil.make_archive("/kaggle/working/clips", "zip", out)
    took = time.time() - t0
    print(f"   {dest.name} in {took/60:.1f} min "
          f"({len([x for x in todo if not (out / f"{x['num']:02d}-{x['slug']}.mp4").exists()])} left)",
          flush=True)
    dest.with_suffix(".meta.yaml").write_text(
        "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
            "platform": "kaggle-free-gpu", "model": "Wan2.1-T2V-1.3B (Apache-2.0)",
            "shot_beat": s["num"], "prompt": s["prompt"], "seed": SEED + s["num"],
            "steps": STEPS, "duration_s": 5, "aspect": "9:16 (480x832)",
            "cost_usd": 0.0, "date": date.today().isoformat(),
        }, sort_keys=False, allow_unicode=True))
    print(f"  \u2713 {dest.name}")
    del frames; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")